# Mangrove Pulse: claim validation gates

This notebook separates three evidence questions: independent dated cover labels, operational observation efficiency and ecological outcomes. All three claims remain unproven; the calculation supplies no observations. The [original executed companion](Mangrove_Pulse_Claim_Validation_Executed.ipynb) records the software and analytical checks.

The Khor Kalba Sentinel-2/GMW result measures reference agreement, not field accuracy. Selected review cases, later-date maps and hypothetical drone calculations cannot establish independent 2018 truth or ecological benefit. This notebook loads no geospatial data.

## Methods and claim boundaries

The embedded `claim_checks.py` and its exact synthetic unit tests are loaded in memory and run with the Python standard library. The gates summarize declared metadata; they do not verify reference independence, measurement quality, or field accuracy. Empty CSV templates are exported without invented sample identifiers, probabilities, labels, or observations.

For a held-out map assessment, draw a probability sample only from the 5,653 south test centres, with known inclusion probabilities. The full AOI includes calibration and excluded centres, so it is a separate descriptive population; excluded centres must never be recoded as non-mangrove. Use independent imagery close to 20 November 2018, blind interpretation, an explicit unknown category, documented source/date/provenance, and design-weighted estimates with uncertainty. The chosen review cases remain exploratory.

Sources: [EAD, 2023 drone mangrove project](https://www.ead.gov.ae/en/Media-Centre/News/0-plants-one-million-mangrove); [Olofsson et al. (2014), accuracy assessment good practices](https://doi.org/10.1016/j.rse.2014.02.015); [UAE Mangrove Restoration Guidelines](https://www.admangroves.ae/media/birkfu4i/mangrove-restoration-guidelines.pdf). EAD's 2023 report describes an existing drone planting effort. It supplies comparison context, not evidence that Mangrove Pulse has equivalent hardware or outcomes.

In [ ]:
CLAIM_CHECKS_SOURCE = "\"\"\"Metadata-only claim gates for future field-observation evidence.\n\nThese helpers summarize declared records; they do not establish measurement\nquality, reference independence, ecological truth, or platform superiority.\n\"\"\"\n\nfrom collections import defaultdict\nfrom datetime import date\nfrom math import isfinite\nfrom numbers import Real\n\n\nTARGET_DATE = \"2018-11-20\"\nLABELS = {\"mangrove\", \"non_mangrove\", \"unknown\"}\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f\"{name} must be a finite nonnegative number\")\n    if not isfinite(value) or value < 0:\n        raise ValueError(f\"{name} must be a finite nonnegative number\")\n    return float(value)\n\n\ndef _count(value, name, minimum=0):\n    if isinstance(value, bool) or not isinstance(value, int) or value < minimum:\n        raise ValueError(f\"{name} must be an integer >= {minimum}\")\n    return value\n\n\ndef accepted_observation_summary(records):\n    \"\"\"Summarize declared accepted observations per resource, separated by kind.\n\n    Each record needs platform, mission_energy_wh, elapsed_min, attempted,\n    accepted, and evidence_kind (``measured`` or ``hypothetical``). Resource\n    totals must include failed attempts and recovery costs. Ratios are\n    descriptive only and cannot prove superiority or equivalent quality.\n    \"\"\"\n    groups = defaultdict(lambda: {\"energy\": 0.0, \"minutes\": 0.0,\n                                 \"attempted\": 0, \"accepted\": 0})\n    for index, record in enumerate(records):\n        try:\n            platform = record[\"platform\"]\n            kind = record[\"evidence_kind\"]\n            energy = _finite_nonnegative(record[\"mission_energy_wh\"], \"mission_energy_wh\")\n            minutes = _finite_nonnegative(record[\"elapsed_min\"], \"elapsed_min\")\n            attempted = _count(record[\"attempted\"], \"attempted\")\n            accepted = _count(record[\"accepted\"], \"accepted\")\n        except (KeyError, TypeError) as exc:\n            raise ValueError(f\"record {index} is missing a required field\") from exc\n        if not isinstance(platform, str) or not platform.strip():\n            raise ValueError(\"platform must be a non-empty string\")\n        if kind not in {\"measured\", \"hypothetical\"}:\n            raise ValueError(\"evidence_kind must be measured or hypothetical\")\n        if accepted > attempted:\n            raise ValueError(\"accepted cannot exceed attempted\")\n        if accepted > 0 and energy == 0:\n            raise ValueError(\"positive accepted observations require positive energy\")\n        if accepted > 0 and minutes == 0:\n            raise ValueError(\"positive accepted observations require positive elapsed time\")\n        group = groups[(platform, kind)]\n        group[\"energy\"] += energy\n        group[\"minutes\"] += minutes\n        group[\"attempted\"] += attempted\n        group[\"accepted\"] += accepted\n\n    result = []\n    for (platform, kind), totals in sorted(groups.items()):\n        accepted = totals[\"accepted\"]\n        energy, minutes = totals[\"energy\"], totals[\"minutes\"]\n        result.append({\n            \"platform\": platform,\n            \"evidence_kind\": kind,\n            \"attempted\": totals[\"attempted\"],\n            \"accepted\": accepted,\n            \"mission_energy_wh\": energy,\n            \"elapsed_min\": minutes,\n            \"accepted_per_wh\": accepted / energy if energy else None,\n            \"accepted_per_min\": accepted / minutes if minutes else None,\n            \"descriptive_only\": True,\n            \"proves_superiority\": False,\n        })\n    return result\n\n\ndef adjudication_gate(records):\n    \"\"\"Check whether declared reference-review metadata is ready for review.\n\n    This deliberately does not compute field accuracy. ``independent_reference``\n    is a self-declaration to be verified, never proof of independence.\n    \"\"\"\n    rows = list(records)\n    ids = set()\n    counts = {label: 0 for label in LABELS}\n    counts[\"missing_or_unlabelled\"] = 0\n    issues = [] if rows else [\"no adjudication records supplied\"]\n    review_flags = []\n    for index, row in enumerate(rows):\n        if not isinstance(row, dict):\n            issues.append(f\"record {index}: record must be a mapping\")\n            continue\n        sample_id = row.get(\"sample_id\")\n        if not isinstance(sample_id, str) or not sample_id.strip():\n            issues.append(f\"record {index}: missing sample_id\")\n        elif sample_id in ids:\n            raise ValueError(f\"duplicate sample_id: {sample_id}\")\n        else:\n            ids.add(sample_id)\n\n        label = row.get(\"reference_label\")\n        if isinstance(label, str) and label in LABELS:\n            counts[label] += 1\n        else:\n            counts[\"missing_or_unlabelled\"] += 1\n            issues.append(f\"record {index}: missing or invalid reference_label\")\n\n        probability = row.get(\"inclusion_probability\")\n        try:\n            if isinstance(probability, bool) or not isinstance(probability, Real) or not isfinite(probability) or not 0 < probability <= 1:\n                raise ValueError\n        except (TypeError, ValueError):\n            issues.append(f\"record {index}: inclusion_probability must satisfy 0 < p <= 1\")\n\n        target_date = row.get(\"target_date\")\n        try:\n            if not isinstance(target_date, str) or date.fromisoformat(target_date).isoformat() != target_date:\n                raise ValueError\n        except ValueError:\n            issues.append(f\"record {index}: target_date must be a valid ISO date\")\n        if target_date != TARGET_DATE:\n            issues.append(f\"record {index}: target_date must be {TARGET_DATE}\")\n        prediction = row.get(\"prediction_class\")\n        if not isinstance(prediction, str) or prediction not in {\"mangrove\", \"non_mangrove\"}:\n            issues.append(f\"record {index}: missing or invalid prediction_class\")\n        if not isinstance(row.get(\"source_url\"), str) or not row[\"source_url\"].strip():\n            issues.append(f\"record {index}: source_url required\")\n        reference_date = row.get(\"reference_date\")\n        try:\n            if not isinstance(reference_date, str) or date.fromisoformat(reference_date).isoformat() != reference_date:\n                raise ValueError\n        except ValueError:\n            issues.append(f\"record {index}: reference_date must be a valid ISO date\")\n        else:\n            if reference_date != TARGET_DATE:\n                review_flags.append(f\"record {index}: reference_date differs from target_date\")\n        if not isinstance(row.get(\"independent_reference\"), bool):\n            issues.append(f\"record {index}: independent_reference boolean required\")\n        elif not row[\"independent_reference\"]:\n            issues.append(f\"record {index}: reference independence not established\")\n        if not isinstance(row.get(\"blind_interpretation\"), bool):\n            issues.append(f\"record {index}: blind_interpretation boolean required\")\n        elif not row[\"blind_interpretation\"]:\n            issues.append(f\"record {index}: blind interpretation required\")\n    if counts[\"unknown\"] or counts[\"missing_or_unlabelled\"]:\n        issues.append(\"unknown or unlabelled references prevent an accuracy claim\")\n\n    return {\n        \"status\": \"ready_for_review\" if not issues else \"not_ready\",\n        \"counts\": counts,\n        \"records\": len(rows),\n        \"issues\": issues,\n        \"review_flags\": review_flags,\n        \"independence_is_self_declared_not_verified\": True,\n        \"no_accuracy_claim\": True,\n        \"ready_for_accuracy_claim\": False,\n    }\n"
TEST_CLAIM_CHECKS_SOURCE = "\"\"\"Synthetic unit tests for metadata claim gates; no acquired data used.\"\"\"\n\nimport unittest\n\nfrom claim_checks import accepted_observation_summary, adjudication_gate\n\n\nclass AcceptedObservationTests(unittest.TestCase):\n    def row(self, **updates):\n        value = {\"platform\": \"aircraft\", \"mission_energy_wh\": 10.0,\n                 \"elapsed_min\": 2.0, \"attempted\": 3, \"accepted\": 2,\n                 \"evidence_kind\": \"measured\"}\n        value.update(updates)\n        return value\n\n    def test_aggregates_by_platform_and_keeps_kinds_separate(self):\n        result = accepted_observation_summary([\n            self.row(), self.row(mission_energy_wh=5, elapsed_min=1,\n                                 attempted=1, accepted=1),\n            self.row(evidence_kind=\"hypothetical\")])\n        by_kind = {row[\"evidence_kind\"]: row for row in result}\n        measured, hypothetical = by_kind[\"measured\"], by_kind[\"hypothetical\"]\n        self.assertEqual((measured[\"attempted\"], measured[\"accepted\"]), (4, 3))\n        self.assertEqual(measured[\"accepted_per_wh\"], 0.2)\n        self.assertEqual(measured[\"accepted_per_min\"], 1.0)\n        self.assertEqual(hypothetical[\"accepted\"], 2)\n        self.assertFalse(measured[\"proves_superiority\"])\n\n    def test_zero_denominators_and_zero_acceptance(self):\n        row = self.row(mission_energy_wh=0, elapsed_min=0, attempted=0, accepted=0)\n        result = accepted_observation_summary([row])[0]\n        self.assertIsNone(result[\"accepted_per_wh\"])\n        self.assertIsNone(result[\"accepted_per_min\"])\n\n    def test_invalid_resources_and_counts_rejected(self):\n        for field, value in ((\"mission_energy_wh\", float(\"nan\")),\n                             (\"mission_energy_wh\", -1), (\"elapsed_min\", float(\"inf\")),\n                             (\"attempted\", -1), (\"attempted\", True),\n                             (\"accepted\", True), (\"accepted\", 4),\n                             (\"mission_energy_wh\", 0), (\"elapsed_min\", 0)):\n            with self.subTest(field=field, value=value):\n                with self.assertRaises(ValueError):\n                    accepted_observation_summary([self.row(**{field: value})])\n\n    def test_evidence_kinds_are_never_combined(self):\n        result = accepted_observation_summary([\n            self.row(), self.row(evidence_kind=\"hypothetical\")])\n        self.assertEqual({row[\"evidence_kind\"] for row in result},\n                         {\"measured\", \"hypothetical\"})\n\n\nclass AdjudicationTests(unittest.TestCase):\n    def row(self, **updates):\n        value = {\"sample_id\": \"s1\", \"inclusion_probability\": 0.5,\n                 \"reference_label\": \"mangrove\", \"source_url\": \"https://example.org/ref\",\n                 \"reference_date\": \"2018-11-20\", \"independent_reference\": True,\n                 \"blind_interpretation\": True, \"prediction_class\": \"mangrove\",\n                 \"target_date\": \"2018-11-20\"}\n        value.update(updates)\n        return value\n\n    def test_ready_is_not_an_accuracy_claim(self):\n        result = adjudication_gate([self.row()])\n        self.assertEqual(result[\"status\"], \"ready_for_review\")\n        self.assertTrue(result[\"no_accuracy_claim\"])\n        self.assertTrue(result[\"independence_is_self_declared_not_verified\"])\n\n    def test_unknown_is_counted_and_not_dropped(self):\n        result = adjudication_gate([self.row(), self.row(\n            sample_id=\"s2\", reference_label=\"unknown\")])\n        self.assertEqual(result[\"counts\"][\"unknown\"], 1)\n        self.assertEqual(result[\"records\"], 2)\n        self.assertEqual(result[\"status\"], \"not_ready\")\n        self.assertFalse(result[\"ready_for_accuracy_claim\"])\n\n    def test_empty_and_unblinded_or_nonindependent_are_not_ready(self):\n        self.assertEqual(adjudication_gate([])[\"status\"], \"not_ready\")\n        for changes in ({\"independent_reference\": False},\n                        {\"blind_interpretation\": False}):\n            with self.subTest(changes=changes):\n                self.assertEqual(adjudication_gate([self.row(**changes)])[\"status\"],\n                                 \"not_ready\")\n\n    def test_dates_are_iso_and_reference_mismatch_is_only_a_review_flag(self):\n        invalid = adjudication_gate([self.row(target_date=\"2018-02-30\")])\n        self.assertEqual(invalid[\"status\"], \"not_ready\")\n        mismatch = adjudication_gate([self.row(reference_date=\"2017-04-10\")])\n        self.assertEqual(mismatch[\"status\"], \"ready_for_review\")\n        self.assertTrue(mismatch[\"review_flags\"])\n        self.assertFalse(mismatch[\"ready_for_accuracy_claim\"])\n\n    def test_missing_label_or_required_metadata_is_not_ready(self):\n        result = adjudication_gate([self.row(reference_label=None, source_url=\"\")])\n        self.assertEqual(result[\"status\"], \"not_ready\")\n        self.assertEqual(result[\"counts\"][\"missing_or_unlabelled\"], 1)\n\n    def test_duplicate_and_invalid_probability_rejected_or_gated(self):\n        with self.assertRaises(ValueError):\n            adjudication_gate([self.row(), self.row()])\n        for probability in (0, 1.1, float(\"nan\"), -0.1):\n            with self.subTest(probability=probability):\n                result = adjudication_gate([self.row(inclusion_probability=probability)])\n                self.assertEqual(result[\"status\"], \"not_ready\")\n\n    def test_wrong_date_and_prediction_are_not_ready(self):\n        result = adjudication_gate([self.row(target_date=\"2018-11-30\",\n                                             prediction_class=\"uncertain\")])\n        self.assertEqual(result[\"status\"], \"not_ready\")\n        self.assertTrue(result[\"no_accuracy_claim\"])\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n"

import sys
import types
import unittest

claim_checks_module = types.ModuleType("claim_checks")
exec(CLAIM_CHECKS_SOURCE, claim_checks_module.__dict__)
sys.modules["claim_checks"] = claim_checks_module

test_module = types.ModuleType("test_claim_checks")
test_module.__file__ = "test_claim_checks.py (embedded exact source)"
exec(TEST_CLAIM_CHECKS_SOURCE, test_module.__dict__)
test_suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
test_result = unittest.TextTestRunner(verbosity=2).run(test_suite)
assert test_result.wasSuccessful(), "Embedded claim gate tests failed"
print("All embedded claim gate tests passed.")

## Analytical acceptance frontier (hypothetical only)

The algebra uses previously saved rounded illustrative inputs, rather than a physical model rerun or field results. Both options attempt the same hypothetical three stations. Let `pH` and `pS` be the unknown acceptance fractions for hover and supported modes. Ratios of expected resource totals to expected accepted counts are `E / (3p)` for energy and `T / (3p)` for elapsed time. These are not expectations of per-mission ratios, which can be undefined when no observations are accepted.

Energy improves only if `pS/pH > 128/170.7`; time improves only if `pS/pH > 840/750`. Both improve only if the stricter ratio is exceeded. The table gives the minimum `pS` thresholds for each illustrative `pH`; equality is break-even, not strict improvement. These are not UAE product, Khor Kalba route, field validation, or equivalent-quality claims.

In [ ]:
E_HOVER_WH = 170.7
E_SUPPORTED_WH = 128.0
T_HOVER_SECONDS = 750.0
T_SUPPORTED_SECONDS = 840.0
ATTEMPTED_STATIONS = 3

energy_ratio_threshold = E_SUPPORTED_WH / E_HOVER_WH
time_ratio_threshold = T_SUPPORTED_SECONDS / T_HOVER_SECONDS
both_ratio_threshold = max(energy_ratio_threshold, time_ratio_threshold)

print("Illustrative strict-improvement ratio thresholds (pS / pH):")
print(f"  energy: {energy_ratio_threshold:.6f}")
print(f"  elapsed time: {time_ratio_threshold:.6f}")
print(f"  both: {both_ratio_threshold:.6f}")
print("\npH      pS energy   pS time     pS both     status")
for p_hover in (0.5, 0.7, 0.9, 1.0):
    p_supported_energy = p_hover * energy_ratio_threshold
    p_supported_time = p_hover * time_ratio_threshold
    p_supported_both = p_hover * both_ratio_threshold
    impossible = p_supported_both >= 1.0
    status = "strict both-improvement impossible (pS must be <= 1)" if impossible else "possible only above both threshold"
    print(f"{p_hover:.1f}   {p_supported_energy:.6f}   {p_supported_time:.6f}   {p_supported_both:.6f}   {status}")

print("\nPer accepted observation at any chosen p:")
print("pH     hover Wh/accepted  hover sec/accepted")
for p_hover in (0.5, 0.7, 0.9, 1.0):
    print(f"{p_hover:.1f}    {E_HOVER_WH / (ATTEMPTED_STATIONS * p_hover):.3f}             {T_HOVER_SECONDS / (ATTEMPTED_STATIONS * p_hover):.3f}")
print("Supported-mode values require an observed pS; none is supplied here.")

## Blank evidence templates

The final cell creates a fresh results package in a persistent temporary directory. The platform mission, independent-label, prediction-join, and ecological-monitoring CSVs contain headers only. The historical label target is 2018-11-20; operational missions and ecological monitoring require their actual observation dates and are not assigned that historical date. The status JSON reports zero supplied observations and marks all claims unproven. The ZIP includes an explanatory README, the exact gate source and tests, and a SHA-256 manifest that does not hash itself. It does not calculate accuracy or causal benefit.

In [ ]:
import csv
import hashlib
import json
import tempfile
import zipfile
from pathlib import Path

TARGET_DATE = "2018-11-20"
claim_status = {
    "historical_label_target_date": TARGET_DATE,
    "actual_observations": {"independent_labels": 0, "mission_records": 0, "ecological_monitoring_records": 0},
    "claims": {
        "independent_2018_field_accuracy": {"status": "unproven", "basis": "No independent dated reference labels supplied."},
        "operational_observation_advantage": {"status": "unproven", "basis": "No measured platform missions or accepted observations supplied."},
        "ecological_causal_benefit": {"status": "unproven", "basis": "No ecological outcomes or controlled comparison supplied."}
    },
    "accuracy_or_causal_benefit_reported": False
}

readme = """Mangrove Pulse claim-validation templates\n\nAll CSV templates are intentionally blank apart from headers. No samples, IDs, probabilities, labels, mission measurements, or ecological outcomes are fabricated. The historical reference-label target date is 2018-11-20. This date does not apply to current platform missions or ecological monitoring, which must use their actual observation dates.\n\nFor held-out map inference, sample only the 5,653 south test centres with known inclusion probabilities. Full-AOI summaries are separate and include calibration and excluded centres; excluded centres must not be labeled non-mangrove. Keep blind labels separate from the prediction join file until adjudication is complete.\n\nThe included gate code summarizes metadata only. A ready-for-review status is not accuracy validation; independence remains self-declared until verified. No field accuracy or causal ecological benefit is computed. The energy frontier in the notebook is hypothetical algebra using saved rounded illustrative inputs.\n\nResearch references:\n- EAD, 2023 drone mangrove project: https://www.ead.gov.ae/en/Media-Centre/News/0-plants-one-million-mangrove\n- Olofsson et al. (2014): https://doi.org/10.1016/j.rse.2014.02.015\n- UAE Mangrove Restoration Guidelines: https://www.admangroves.ae/media/birkfu4i/mangrove-restoration-guidelines.pdf\n"""

tmp_root = Path(tempfile.mkdtemp(prefix="mangrove-pulse-claim-validation-"))
out_dir = tmp_root / "claim_validation"
out_dir.mkdir()
(out_dir / "README.txt").write_text(readme, encoding="utf-8")
(out_dir / "status.json").write_text(json.dumps(claim_status, indent=2) + "\n", encoding="utf-8")

templates = {
    "platform_missions_blank.csv": ["mission_id", "matched_session_id", "platform", "observation_date", "evidence_kind", "acceptance_rule_id", "calibration_record", "tide_context", "sensor_depth_m", "mission_energy_wh", "elapsed_min", "attempted", "accepted", "failed_attempts_and_recovery_included", "disturbance_notes"],
    "independent_labels_blank.csv": ["sample_id", "target_date", "reference_label", "reference_date", "source_url", "source_provenance", "independent_reference", "reference_resolution_m", "position_accuracy_m", "interpreter_id", "blind_interpretation", "confidence", "notes"],
    "prediction_join_blank.csv": ["sample_id", "prediction_class", "inclusion_probability"],
    "ecological_monitoring_blank.csv": ["plot_id", "treatment_or_control", "observation_date", "baseline_date", "followup_date", "planted_count", "established_count", "surviving_count", "natural_recruitment_count", "mangrove_cover", "habitat_disturbance_type", "habitat_disturbance_severity", "bird_observations", "mudflat_measurement", "mudflat_measurement_unit", "measurement_method", "source_url", "notes"],
}
for filename, headers in templates.items():
    with (out_dir / filename).open("w", newline="", encoding="utf-8") as handle:
        csv.writer(handle).writerow(headers)

(out_dir / "claim_checks.py").write_text(CLAIM_CHECKS_SOURCE, encoding="utf-8")
(out_dir / "test_claim_checks.py").write_text(TEST_CLAIM_CHECKS_SOURCE, encoding="utf-8")
payloads = sorted(path for path in out_dir.iterdir() if path.is_file())
manifest = {"algorithm": "sha256", "files": [{"path": p.name, "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "bytes": p.stat().st_size} for p in payloads]}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
zip_path = tmp_root / "Mangrove_Pulse_Claim_Validation.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out_dir.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Created fresh package: {zip_path.name} ({zip_path.stat().st_size} bytes)")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f"Download fallback: {zip_path}")